# 07 · Inference

Does the released model run on new lesions, including the ways a clinical image can be missing? We run the
15-checkpoint ensemble on six sample lesions, check every output, and compare the predictions with the reference
predictions stored in `results/predictions/`.

_This model was developed for a research competition and is not a medical device._

In [1]:
# Setup: every path comes from configs/final.yaml (milk10k.config); nothing here downloads data.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RELEASE = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(RELEASE / "src") not in sys.path:
    sys.path.insert(0, str(RELEASE / "src"))  # works without `pip install -e .`
from milk10k.config import load_config
import os
import subprocess
import time

import torch

cfg = load_config()
DATA = cfg["paths"]["data_dir"]  # prepared MILK10k (data/README.md); override with MILK10K_DATA_DIR

# "data:<file>" = under DATA; anything else is relative to the release root.
INPUTS = {
    "manifest": "models/final-P1a-kg/manifest.json",
    "tabular_encoding": "models/final-P1a-kg/tabular_encoding.json",
    "sample_ids": "sample/sample_ids.csv",
    "make_sample": "sample/make_sample.py",
    "sample_meta": "sample/sample_metadata.csv",
    "sample_images": "sample/images/",
    "pred_final": "results/predictions/final-P1a-kg.csv",
    "pred_tta": "results/predictions/final-P1a-kg-tta.csv",
    "prior": "data:sample_submission.csv"
}
OUTPUTS = {
    "sample_predictions": "results/sample_predictions.csv",
    "sample_predictions_tta": "results/sample_predictions_tta.csv",
    "inference_report": "results/metrics/sample_inference_report.json"
}


def inp(key):
    v = INPUTS[key]
    return DATA / v[5:] if v.startswith("data:") else RELEASE / v


def out(key):
    p = RELEASE / OUTPUTS[key]
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def savefig(key):
    plt.tight_layout()
    plt.savefig(out(key), dpi=120)
    plt.show()


missing = [k for k in INPUTS if not inp(k).exists()]
import os  # noqa: E402
print("data dir from:", "MILK10K_DATA_DIR" if os.environ.get("MILK10K_DATA_DIR") else "configs/final.yaml")
print("checkpoints from:", "MILK10K_CHECKPOINT_DIR" if os.environ.get("MILK10K_CHECKPOINT_DIR") else "configs/final.yaml")
print("missing inputs:", missing or "none")

data dir from: MILK10K_DATA_DIR
checkpoints from: configs/final.yaml
missing inputs: none


## 1. Are these the released checkpoints?
Every file must match the sha256 recorded in `models/final-P1a-kg/manifest.json`.

In [2]:
from milk10k.evaluation import sha256
from milk10k.inference import PROBS, parity, predict_lesions

manifest = json.loads(inp("manifest").read_text())
ckpt_dir = cfg["paths"]["checkpoint_dir"]
bad = [c["file"] for c in manifest["checkpoints"]
       if not (ckpt_dir / c["file"]).is_file() or sha256(ckpt_dir / c["file"]) != c["sha256"]]
assert not bad, f"missing or modified checkpoints: {bad} (see models/final-P1a-kg/README.md)"
print(f"{manifest['members']} checkpoints verified (sha256); manifest cross-checks passed: {manifest['all_checks_passed']}")

<release>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


15 checkpoints verified (sha256); manifest cross-checks passed: True


## 2. Six sample lesions
`sample/sample_ids.csv` lists six official **test** lesions: the first six `isic_id` values in sorted order, chosen
without looking at images or predictions. `make_sample.py` copies their images from the local dataset if they are not
there yet. Three lesions are intact. In the other three the clinical image is blank, NaN or a path to a missing file.
A CSV round trip turns `""` into NaN, so the two blank scenarios are re-applied explicitly.

In [3]:
if not inp("sample_meta").is_file():
    subprocess.run([sys.executable, str(inp("make_sample"))], check=True,
                   env={**os.environ, "MILK10K_DATA_DIR": str(DATA)})
sample = pd.read_csv(inp("sample_meta"))
sample["clinical_image_path"] = sample["clinical_image_path"].astype(object)
sample.loc[sample.scenario == "clinical_blank", "clinical_image_path"] = ""
sample.loc[sample.scenario == "clinical_nan", "clinical_image_path"] = np.nan
sample[["isic_id", "scenario", "clinical_image_path", "clinical_available", "age_approx", "sex", "fitzpatrick_type"]]

,isic_id,scenario,clinical_image_path,clinical_available,age_approx,sex,fitzpatrick_type
0,ISIC_0145429,intact,images/test/clinical/ISIC_7021698.jpg,1,60.0,male,5
1,ISIC_0150253,intact,images/test/clinical/ISIC_7519566.jpg,1,60.0,male,4
2,ISIC_0168990,intact,images/test/clinical/ISIC_2499453.jpg,1,65.0,female,4
3,ISIC_0208181,clinical_blank,,0,75.0,male,5
4,ISIC_0241487,clinical_nan,NaN,0,65.0,male,4
5,ISIC_0243690,clinical_missing_file,images/test/clinical/DOES_NOT_EXIST.jpg,0,85.0,male,5


## 3. Predict
The main path uses the primary model, without TTA. If a row's dermoscopic image cannot be read, it receives the
training-set class prior instead of crashing the run.

In [4]:
USE_TTA = cfg["inference"]["tta"]
if inp("prior").is_file():
    prior = pd.read_csv(inp("prior")).iloc[0][PROBS].astype(float)  # training-set class priors (data_prep)
else:
    prior = pd.Series([0.5] + [1 / 11] * 11, index=PROBS)
    print("WARNING: sample_submission.csv not found; a uniform prior is used for unreadable-derm rows")
t0 = time.time()
pred, report = predict_lesions(sample.drop(columns=["scenario"]), RELEASE / "sample", cfg, prior, tta=USE_TTA)
print(f"{len(pred)} lesions in {time.time() - t0:.1f}s on {'cuda' if torch.cuda.is_available() else 'cpu'} (TTA={USE_TTA})")
print(json.dumps(report, indent=1))
pred.merge(sample[["isic_id", "scenario"]], on="isic_id").round(4)

6 lesions in 8.9s on cuda (TTA=False)
{
 "rows_in": 6,
 "duplicate_ids_dropped": 0,
 "clinical_blank_or_nan": 2,
 "clinical_missing_or_unreadable_file": 1,
 "derm_missing_or_unreadable_file": 0,
 "fitzpatrick_out_of_range_set_missing": 0,
 "rows_out": 6,
 "rows_repaired_before_output": 0,
 "checks": {
  "rows_equal_input_ids": true,
  "ids_in_input_order": true,
  "no_duplicate_ids": true,
  "columns": true,
  "finite_no_nan": true,
  "in_unit_interval": true,
  "class_sums_within_1e-3": true,
  "scorer_validate": true
 },
 "tta": false
}


,isic_id,target_binary,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC,scenario
0,ISIC_0145429,0.8441,0.1066,0.1964,0.0051,0.0641,0.0006,0.0043,0.0004,0.0256,0.0038,0.5778,0.0155,intact
1,ISIC_0150253,0.8276,0.1164,0.3057,0.0006,0.1004,0.0040,0.0019,0.0006,0.0137,0.0047,0.4455,0.0064,intact
2,ISIC_0168990,0.5132,0.1544,0.3820,0.0015,0.2698,0.0434,0.0054,0.0007,0.0220,0.0263,0.0935,0.0011,intact
3,ISIC_0208181,0.5958,0.0301,0.4274,0.0674,0.0402,0.0155,0.3112,0.0089,0.0122,0.0181,0.0432,0.0258,clinical_blank
4,ISIC_0241487,0.7376,0.0709,0.5200,0.0015,0.2788,0.0133,0.0029,0.0113,0.0295,0.0320,0.0389,0.0008,clinical_nan
5,ISIC_0243690,0.5892,0.2510,0.0498,0.0008,0.1939,0.0022,0.0006,0.0000,0.0243,0.0051,0.4711,0.0011,clinical_missing_file


All six lesions get a prediction, and every output check passes. The three missing-clinical forms are all recognised
(two blank or NaN, one missing file) and routed to the `[NULL]` token.

## 4. Do we reproduce the reference predictions?
The intact lesions also appear in `results/predictions/`, which were produced on a T4 GPU with torch 2.11 in fp16.
A different GPU and torch build change fp16 arithmetic slightly, so we allow a tolerance of 0.002
(`inference.parity_tolerance`) instead of demanding identical bits.

In [5]:
reference = pd.read_csv(inp("pred_tta" if USE_TTA else "pred_final"))
intact = sample.isic_id[sample.scenario == "intact"]
par = parity(pred[pred.isic_id.isin(intact)], reference, cfg["inference"]["parity_tolerance"])
par

{'checked': True,
 'rows': 3,
 'max_abs_diff': 8.602678612013026e-05,
 'exact': False,
 'within_tolerance': True,
 'tolerance': 0.002}

The largest difference is 8.6e-05, far inside the tolerance: the released model reproduces its reference predictions.

## 5. The optional TTA variant
The same 15 checkpoints, averaged over four flip views of each lesion. We check its parity against the stored TTA
predictions with the same tolerance.

In [6]:
pred_tta, report_tta = predict_lesions(sample.drop(columns=["scenario"]), RELEASE / "sample", cfg, prior, tta=True)
par_tta = parity(pred_tta[pred_tta.isic_id.isin(intact)], pd.read_csv(inp("pred_tta")),
                 cfg["inference"]["parity_tolerance"])
print("TTA output checks passed:", all(report_tta["checks"].values()))
par_tta

TTA output checks passed: True


{'checked': True,
 'rows': 3,
 'max_abs_diff': 4.211962223049648e-05,
 'exact': False,
 'within_tolerance': True,
 'tolerance': 0.002}

The TTA variant also passes every output check, and stays within 4.2e-05 of its stored reference predictions.

## 6. What does a missing clinical image change?
We take the three intact lesions again and blank their clinical path, so that the `[NULL]` token replaces the clinical view.

In [7]:
blanked = sample[sample.scenario == "intact"].drop(columns=["scenario"]).assign(clinical_image_path="")
pred_blank, _ = predict_lesions(blanked, RELEASE / "sample", cfg, prior, tta=USE_TTA)
effect = pred[pred.isic_id.isin(intact)].set_index("isic_id")[["target_binary"]].join(
    pred_blank.set_index("isic_id")[["target_binary"]], rsuffix="_without_clinical")
effect["change"] = effect.target_binary_without_clinical - effect.target_binary
effect.round(4)

,target_binary,target_binary_without_clinical,change
isic_id,,,
ISIC_0145429,0.8441,0.8599,0.0158
ISIC_0150253,0.8276,0.8373,0.0096
ISIC_0168990,0.5132,0.5217,0.0085


Without the clinical image, P(malignant) rises by 0.0085 to 0.0158 on these three lesions. The prediction stays
usable, but the clinical view does carry information (notebook 05 measures that on all lesions).

## 7. Do malformed rows crash inference?
We build synthetic rows in memory: an unreadable dermoscopic image, an out-of-range skin-tone code, an unseen site, and a duplicate ID.

In [8]:
base = sample[sample.scenario == "intact"].head(1).drop(columns=["scenario"])
broken = pd.concat([base.assign(isic_id="SYNTH_derm_missing", derm_image_path="images/test/derm/DOES_NOT_EXIST.jpg"),
                    base.assign(isic_id="SYNTH_fst_out_of_range", fitzpatrick_type=9),
                    base.assign(isic_id="SYNTH_unseen_site", anatomical_site_general="unseen site"),
                    base.assign(isic_id="SYNTH_fst_out_of_range", fitzpatrick_type=9)], ignore_index=True)
pred_broken, rep_broken = predict_lesions(broken, RELEASE / "sample", cfg, prior, tta=False)
print(json.dumps({k: v for k, v in rep_broken.items() if k != "checks"}, indent=1))
pred_broken.round(4)

{
 "rows_in": 4,
 "duplicate_ids_dropped": 1,
 "clinical_blank_or_nan": 0,
 "clinical_missing_or_unreadable_file": 0,
 "derm_missing_or_unreadable_file": 1,
 "fitzpatrick_out_of_range_set_missing": 1,
 "rows_out": 3,
 "rows_repaired_before_output": 0,
 "tta": false
}


,isic_id,target_binary,AKIEC,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
0,SYNTH_derm_missing,0.6592,0.0578,0.4813,0.0084,0.1038,0.0099,0.0095,0.0017,0.0859,0.1424,0.0903,0.0090
1,SYNTH_fst_out_of_range,0.6999,0.1549,0.2206,0.0196,0.0800,0.0003,0.0120,0.0005,0.0305,0.0056,0.4302,0.0458
2,SYNTH_unseen_site,0.7532,0.0833,0.2385,0.0252,0.0802,0.0015,0.0221,0.0005,0.0384,0.0115,0.4537,0.0452


No crash. The duplicate is dropped and the out-of-range code becomes missing. The row with an unreadable image
receives the training prior (P(malignant) 0.659, the training malignant rate), and the output passes every check.

## 8. Save

In [9]:
pred.to_csv(out("sample_predictions"), index=False)
pred_tta.to_csv(out("sample_predictions_tta"), index=False)
out("inference_report").write_text(json.dumps({"tta": USE_TTA, "report": report, "parity": par, "tta_variant_parity": par_tta,
                                                "malformed_rows_report": rep_broken}, indent=1, default=str) + "\n")
print("saved", OUTPUTS["sample_predictions"], "and", OUTPUTS["inference_report"])

saved results/sample_predictions.csv and results/metrics/sample_inference_report.json


## Takeaways
- The checkpoints match the manifest, and the primary model reproduces its reference predictions within 8.6e-05.
- The TTA variant reproduces its reference too, within 4.2e-05 (section 5).
- Every form of a missing clinical image is handled by the `[NULL]` path, and malformed rows never crash inference.

This closes the series: notebooks 01–07 go from the raw data to a verified, reproducible model.